# L4b Algorithm: Breadth-First Search

Breadth-first search (BFS) explores a directed graph in layers: it processes the starting vertex, then every vertex one edge away, then every newly discovered vertex two edges away, and so on. A first-in, first-out queue preserves that discovery order.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __Trace a queue-based traversal:__ Follow the queue, visited set, and first-visit order after each processed vertex. Distinguish vertices waiting in the queue from vertices whose outgoing edges have already been examined.
> * __Explain discovery-time marking:__ Show why marking a vertex when it enters the queue prevents two vertices from enqueuing the same neighbor. Use that rule to explain why every reachable vertex is eventually processed once.
> * __Explain computational cost:__ Relate the vertices and edges examined during traversal to its running time, and identify the storage needed to track the search.

We trace the queue on the lab graph, explain why discovery-time marking avoids repeated work, and examine the time and storage required by the algorithm.

Let's get started!

___

## The Algorithm

The figure shows the directed graph used in the L4b lab. Its adjacency list contains `1 => [2, 3]`, so processing vertex 1 discovers vertex 2 before vertex 3.

<div>
    <center>
        <img src="figs/Fig-Example-Graph.svg" width="480" alt="Six-vertex directed graph used for the breadth-first-search trace"/>
    </center>
</div>

The edge weights shown in the figure do not affect this traversal. Layers count edges, not total edge weight.

### Algorithm: Breadth-first search

__Initialization__: Given a graph $\mathcal{G}=(\mathcal{V},\mathcal{E})$ and a starting vertex $v_s\in\mathcal{V}$, initialize an empty set of visited vertices $\mathcal{V}_{\text{visited}}$, an empty queue $\mathcal{Q}$, and an empty visit-order list $\texttt{order}$.

1. Mark the starting vertex as visited: $\mathcal{V}_{\text{visited}}\gets\mathcal{V}_{\text{visited}}\cup\{v_s\}$.
2. Add the starting vertex $v_s$ to the queue: $\mathcal{Q}\gets\texttt{enqueue}(\mathcal{Q},v_s)$.
3. While the queue $\mathcal{Q}$ is not empty, __do__:
    - Dequeue a vertex $v_n$ from the front of the queue: $v_n\gets\texttt{dequeue}(\mathcal{Q})$.
    - Append $v_n$ to the visit-order list $\texttt{order}$.
    - Get the outgoing neighbors of $v_n$ from the adjacency list: $\mathcal{N}_n\gets\texttt{neighbors}(v_n)$.
    - For each neighbor $v_m\in\mathcal{N}_n$, in ascending identifier order, __do__:
        - If $v_m\notin\mathcal{V}_{\text{visited}}$, then:
            - Mark $v_m$ as visited: $\mathcal{V}_{\text{visited}}\gets\mathcal{V}_{\text{visited}}\cup\{v_m\}$.
            - Enqueue it at the back: $\mathcal{Q}\gets\texttt{enqueue}(\mathcal{Q},v_m)$.
4. Return $\texttt{order}$ when the queue is empty.

__Why mark a vertex when it enters the queue?__ Suppose vertices 1 and 2 both point to vertex 3. Once vertex 1 adds vertex 3 to the queue, marking it visited prevents vertex 2 from adding it again.

> __What the queue contains:__
>
> At the start of each iteration, every queued vertex has been discovered, but its outgoing edges have not yet been examined. Removing from the front processes vertices in discovery order. New neighbors join the back, so the search completes one layer before processing the next.
>
> The starting vertex forms layer 0. Processing it discovers its unvisited outgoing neighbors in layer 1. Processing the vertices in layer 1 discovers any still-unvisited vertices in layer 2, and we continue in the same way. A vertex’s layer is the minimum number of directed edges needed to reach it from the start.

Marking and enqueuing happen together, so each reachable vertex enters the queue only once.

___

## Trace the Queue

Starting at vertex 1 gives the following state transitions. The queue column records only vertices still waiting to be processed; the order column records vertices after they leave the queue.

| Step | Processed vertex | Newly enqueued | Queue after the step | Visited after the step | Visit order |
|:--|:--:|:--|:--|:--|:--|
| Initialize | none | `1` | `[1]` | `{1}` | `[]` |
| 1 | `1` | `2, 3` | `[2, 3]` | `{1, 2, 3}` | `[1]` |
| 2 | `2` | `4` | `[3, 4]` | `{1, 2, 3, 4}` | `[1, 2]` |
| 3 | `3` | `5` | `[4, 5]` | `{1, 2, 3, 4, 5}` | `[1, 2, 3]` |
| 4 | `4` | `6` | `[5, 6]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 4]` |
| 5 | `5` | none | `[6]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 4, 5]` |
| 6 | `6` | none | `[]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 4, 5, 6]` |

At step 2, vertex 3 is already visited because it entered the queue during step 1, so the edge `2 → 3` does not enqueue it again. At step 5, the same rule ignores `5 → 4`. The same visited-set check prevents repeated processing if an edge leads back around a cycle.

__Does the visit order describe a path?__ The resulting sequence `[1, 2, 3, 4, 5, 6]` is a first-visit order, not a directed path. For example, vertices 3 and 4 are consecutive in the sequence even though the graph has no edge `3 → 4`.

__How do the layers appear in the trace?__ Starting with vertex 1, the search discovers vertices 2 and 3, then vertices 4 and 5, and finally vertex 6. The queue keeps each layer ahead of the vertices discovered from it.

___


## Computational Cost

Let $V_r$ be the set of vertices reachable from the start, and let $E_r$ contain their outgoing directed edges. Assume that the adjacency list has already been built and its neighbor lists ordered.

Each reachable vertex enters and leaves the queue once. Examining the neighbor lists performs one visited-set check for each reachable edge. With constant-time queue operations and expected constant-time dictionary and visited-set operations, the expected running time is given by:
$$
T=\mathcal{O}(|V_r|+|E_r|).
$$
This bound counts the traversal itself; sorting neighbor lists is additional work.

The visited set, queue, and visit-order list each hold at most $|V_r|$ vertex identifiers. Excluding the graph, the additional storage is therefore given by:
$$
S=\mathcal{O}(|V_r|).
$$
The queue need not hold every reachable vertex at once, but the visited set and returned order retain every vertex the search discovers.

___


## Summary

Breadth-first search uses a FIFO queue to explore the graph in layers measured by the number of edges from the starting vertex.

> __Key Takeaways:__
>
> * **Queue order:** We traced how newly discovered vertices join the back of the queue while processing begins at the front. This order lets us finish one layer before advancing to the next.
> * **Discovery-time marking and computational cost:** We marked each vertex when it entered the queue, preventing repeated processing through converging edges or cycles. Counting the reachable vertices and their outgoing edges gave expected $\mathcal{O}(|V_r|+|E_r|)$ traversal time under the stated assumptions. The visited set, queue, and returned order required $\mathcal{O}(|V_r|)$ additional storage.
> * **Interpreting visit order:** We distinguished the order in which vertices are processed from a directed path through the graph. A traversal order records the search process; consecutive entries need not be joined by a directed edge.

Together, the queue and visited set let us explore every reachable vertex once, layer by layer.

___